### Prueba Factores elevación


In [1]:
import numpy as np
import pandas as pd

class RedTransporte:
    def __init__(self, matriz_directos, nombres_estaciones):
        """
        Inicializa la red de transporte con:
        - matriz_directos: Matriz de viajes directos entre estaciones
        - nombres_estaciones: Lista con los nombres de las estaciones
        """
        self.n = len(nombres_estaciones)
        self.nombres = nombres_estaciones
        self.matriz_directos = np.array(matriz_directos)
        
        # Validación de la matriz
        if self.matriz_directos.shape != (self.n, self.n):
            raise ValueError("La matriz de viajes directos debe ser cuadrada de tamaño n x n")
            
        # Inicializar matrices de transbordos
        self.matriz_1_transbordo = None
        self.matriz_2_transbordos = None
        self.matriz_total = None
        
        # Factores de elevación
        self.factores_elevacion = None
    
    def calcular_matrices_transbordo(self):
        """Calcula las matrices con 1 y 2 transbordos"""
        # Matriz con 1 transbordo (producto matricial)
        self.matriz_1_transbordo = np.dot(self.matriz_directos, self.matriz_directos)
        
        # Matriz con 2 transbordos (producto matricial de 1 transbordo con directos)
        self.matriz_2_transbordos = np.dot(self.matriz_1_transbordo, self.matriz_directos)
        
        # Matriz total (suma de todas las posibilidades)
        self.matriz_total = self.matriz_directos + self.matriz_1_transbordo + self.matriz_2_transbordos
    
    def calcular_factores_elevacion(self):
        """Calcula los factores de elevación para subidas, bajadas y transbordos"""
        if self.matriz_total is None:
            self.calcular_matrices_transbordo()
            
        # Subidas (suma por columnas)
        subidas = self.matriz_total.sum(axis=0)
        
        # Bajadas (suma por filas)
        bajadas = self.matriz_total.sum(axis=1)
        
        # Coeficiente de transbordo
        # Calculamos el número de viajes con transbordo (1 o 2 transbordos)
        viajes_transbordo = self.matriz_1_transbordo + self.matriz_2_transbordos
        total_viajes = self.matriz_total.sum()
        
        if total_viajes > 0:
            coeficiente_transbordo = viajes_transbordo.sum() / total_viajes
        else:
            coeficiente_transbordo = 0
            
        self.factores_elevacion = {
            'subidas': subidas,
            'bajadas': bajadas,
            'coeficiente_transbordo': coeficiente_transbordo,
            'matriz_total': self.matriz_total
        }
        
        return self.factores_elevacion
    
    def mostrar_resultados(self):
        """Muestra los resultados de forma legible"""
        if self.factores_elevacion is None:
            self.calcular_factores_elevacion()
            
        print("\n=== FACTORES DE ELEVACIÓN ===")
        
        # Crear DataFrame para subidas y bajadas
        df = pd.DataFrame({
            'Estación': self.nombres,
            'Subidas': self.factores_elevacion['subidas'],
            'Bajadas': self.factores_elevacion['bajadas']
        })
        
        print("\nSubidas y Bajadas por Estación:")
        print(df.to_string(index=False))
        
        print(f"\nCoeficiente de Transbordo: {self.factores_elevacion['coeficiente_transbordo']:.4f}")
        
        print("\nMatriz Total de Viajes:")
        matriz_df = pd.DataFrame(self.factores_elevacion['matriz_total'], 
                                index=self.nombres, 
                                columns=self.nombres)
        print(matriz_df)


# Ejemplo de uso
if __name__ == "__main__":
    # Datos de ejemplo
    nombres_estaciones = ['A', 'B', 'C', 'D']
    
    # Matriz de viajes directos (A -> B, etc.)
    matriz_directos = [
        [0, 100, 0, 0],    # Desde A
        [50, 0, 80, 0],    # Desde B
        [0, 40, 0, 60],    # Desde C
        [0, 0, 30, 0]      # Desde D
    ]
    
    # Crear red
    red = RedTransporte(matriz_directos, nombres_estaciones)
    
    # Calcular y mostrar resultados
    red.calcular_factores_elevacion()
    red.mostrar_resultados()


=== FACTORES DE ELEVACIÓN ===

Subidas y Bajadas por Estación:
Estación  Subidas  Bajadas
       A   477050  1313100
       B  1229540  1223130
       C   963110   707100
       D   786660   213030

Coeficiente de Transbordo: 0.9999

Matriz Total de Viajes:
        A       B       C       D
A    5000  820100    8000  480000
B  410050    8200  800080    4800
C    2000  400040    5000  300060
D   60000    1200  150030    1800
